In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import scipy.stats as stats

df = pd.read_csv("../data/processed/data_before_reduction.csv")
df.head()

## **Câu 3: Giá sản phẩm (price) và tỷ lệ giảm giá (discount_rate) ảnh hưởng như thế nào đến số lượng bán ra (quantity_sold)?**

Mục tiêu: 
- Phân tích mối quan hệ giữa giá sản phẩm, tỷ lệ giảm giá và số lượng bán ra. Chúng ta sẽ sử dụng các phương pháp thống kê mô tả và trực quan hóa dữ liệu để trả lời câu hỏi này. 
- Thống kê xem discount_rate thì có bán được nhiều hơn so với những không discount trong cùng 1 danh mục
 

Để trả lời câu hỏi này, nhóm em sẽ phân tích mối quan hệ giữa giá sản phẩm (price), tỷ lệ giảm giá (discount_rate) và số lượng bán ra (quantity_sold) bằng cách sử dụng các biểu đồ trực quan hóa dữ liệu và các phương pháp thống kê. Chúng ta sẽ xem xét các yếu tố sau:
1. **Phân tích mô tả**: Tính toán các thống kê mô tả như trung bình, trung vị, độ lệch chuẩn của các biến price, discount_rate và quantity_sold để hiểu rõ hơn về phân phối dữ liệu.
2. **Biểu đồ Heatmap**: Vẽ biểu đồ heatmap để quan sát mối quan hệ giữa price và quantity_sold, cũng như giữa discount_rate và quantity_sold tổng quát với toàn bộ sản phẩm.
3. **Biểu đồ scatter (Scatter Plot)**: Vẽ biểu đồ phân tán để quan sát mối quan hệ giữa price và quantity_sold theo nhóm danh mục kèm **đường xu hướng**.
4. **Biểu đồ cột (Bar Chart)**: So sánh số lượng bán ra trung bình giữa các nhóm sản phẩm có và không có giảm giá trong cùng một danh mục.
5. **Kiểm định thống kê**: Sử dụng các phương pháp kiểm định thống kê như hồi quy tuyến tính để xác định mức độ ảnh hưởng của price và discount_rate đến quantity_sold theo từng nhóm danh mục.

In [ ]:
# 1. Phân tích mô tả
desc = df[['price', 'discount_rate', 'quantity_sold']].describe()
print("Mô tả thống kê (với price và quantity_sold đã Log Transform) :")
display(desc)

In [ ]:
# 2. Vẽ heatmap tương quan giữa 3 biến với toàn bộ sản phẩm
columns_for_corr = ['price', 'discount_rate', 'quantity_sold']
corr_matrix = df[columns_for_corr].corr(method='spearman')

plt.figure(figsize=(10, 8))

# Tạo mask để chỉ hiện nửa dưới của heatmap
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, 
            mask=mask,  # Thêm mask để chỉ hiện nửa dưới
            annot=True, 
            fmt='.3f', 
            cmap='RdBu_r',  # Red-Blue colormap
            vmin=-1, vmax=1,
            square=True,
            linewidths=0.5,
            cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix: Price, Discount Rate, and Quantity Sold\n(Toàn bộ sản phẩm)', 
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Variables', fontsize=12)
plt.ylabel('Variables', fontsize=12)
plt.tight_layout()
plt.show()

**Nhận định sơ bộ**: 

Dựa trên Heatmap trên và hệ số tương quan Spearman, ta thấy:
- Có một mối quan hệ nghịch giữa giá sản phẩm (price) và số lượng bán ra (quantity_sold), nghĩa là khi giá sản phẩm tăng thì số lượng bán ra có xu hướng giảm. 
- Tỷ lệ giảm giá (discount_rate) có mối quan hệ thuận với số lượng bán ra, tức là khi tỷ lệ giảm giá tăng thì số lượng bán ra cũng tăng theo. 
=> Mặc dù vậy, để có cái nhìn chính xác hơn, chúng ta cần phân tích sâu hơn qua các biểu đồ scatter và kiểm định thống kê để xác định mức độ ảnh hưởng cụ thể của từng yếu tố đến số lượng bán ra trong từng nhóm danh mục sản phẩm. 

In [ ]:
# 3. Vẽ Scatterplot phân bố quantity_sold theo price theo từng danh mục
# Lọc các danh mục có đủ dữ liệu
categories = [cat for cat in df['category_root_name'].unique() 
              if len(df[df['category_root_name'] == cat]) > 30]

# Thiết lập lưới subplot
import math
n_cats = len(categories)
n_cols = 3  # 3 cột
n_rows = math.ceil(n_cats / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 5))
axes = axes.flatten()  # Làm phẳng mảng axes

# Vẽ scatter plot cho từng danh mục
for i, category in enumerate(categories):
    subset = df[df['category_root_name'] == category]
    
    axes[i].scatter(subset['price'], subset['quantity_sold'], 
                   alpha=0.6, s=30, color='steelblue')
    
    # Vẽ đường xu hướng (trendline)
    if len(subset) > 1:
        z = np.polyfit(subset['price'], subset['quantity_sold'], 1)
        p = np.poly1d(z)
        axes[i].plot(subset['price'].sort_values(), p(subset['price'].sort_values()), 
                    "r--", alpha=0.8, linewidth=2, label='Trend')
    
    axes[i].set_title(f'{category}\n(N={len(subset)})', 
                     fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Price (Log-transformed)', fontsize=10)
    axes[i].set_ylabel('Quantity Sold (Log-transformed)', fontsize=10)
    axes[i].grid(True, alpha=0.3)
    
    # Tính và hiển thị correlation
    corr = subset['price'].corr(subset['quantity_sold'], method='spearman')
    axes[i].text(0.05, 0.95, f'r = {corr:.3f}', 
                transform=axes[i].transAxes, 
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
                fontsize=10, fontweight='bold')

# Xóa các subplot thừa
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Scatter Plot: Quantity Sold vs Price theo từng Danh mục', 
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

**Nhận định**:
- Đa phần các danh mục sản phẩm đều cho thấy mối quan hệ nghịch giữa giá sản phẩm và số lượng bán ra, dựa vào đường xu hướng chếch xuống dưới trong biểu đồ scatter plot.
- Một vài danh mục có thể không tuân theo xu hướng này mà có đường xu hướng nằm ngang. Điều này có thể do đặc thù của sản phẩm hoặc các yếu tố khác ảnh hưởng đến quyết định mua hàng của khách hàng. Các danh mục đó bao gồm: **Túi thời trang nữ, Chăm sóc nhà cửa, Nhà sách Tiki, Điện thoại - Máy tính bảng, Đồ chơi - Mẹ & Bé, Điện Gia Dụng**

In [ ]:
# 3. Vẽ bar plot so sánh có discount và không có discount
# Tạo biến nhị phân cho discount
df['has_discount'] = df['discount_rate'] > 0

# Tính toán cho từng danh mục
categories = [cat for cat in df['category_root_name'].unique() 
              if len(df[df['category_root_name'] == cat]) > 30]

comparison_data = []
for category in categories:
    subset = df[df['category_root_name'] == category]
    
    # Tính trung bình quantity_sold cho có discount và không có discount
    with_discount = subset[subset['has_discount']]['quantity_sold'].mean()
    without_discount = subset[~subset['has_discount']]['quantity_sold'].mean()
    
    comparison_data.append({
        'Category': category,
        'Với Discount': with_discount,
        'Không Discount': without_discount,
        'Chênh lệch': with_discount - without_discount
    })

comparison_df = pd.DataFrame(comparison_data)

# Vẽ bar plot so sánh
x = np.arange(len(categories))
width = 0.35

plt.figure(figsize=(14, 8))
bars1 = plt.bar(x - width/2, comparison_df['Với Discount'], width, 
               label='Với Discount', color='lightgreen', alpha=0.8)
bars2 = plt.bar(x + width/2, comparison_df['Không Discount'], width, 
               label='Không Discount', color='lightcoral', alpha=0.8)

plt.title('So sánh Quantity Sold: Có Discount vs Không Discount', 
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Danh mục sản phẩm', fontsize=12)
plt.ylabel('Trung bình Quantity Sold (Log-transformed)', fontsize=12)
plt.xticks(x, categories, rotation=45, ha='right')
plt.legend(fontsize=12)
plt.grid(axis='y', alpha=0.3)

# Thêm giá trị lên bars
for i, (bar1, bar2) in enumerate(zip(bars1, bars2)):
    plt.text(bar1.get_x() + bar1.get_width()/2, bar1.get_height() + 0.01,
             f'{comparison_df.iloc[i]["Với Discount"]:.2f}', 
             ha='center', va='bottom', fontsize=9)
    plt.text(bar2.get_x() + bar2.get_width()/2, bar2.get_height() + 0.01,
             f'{comparison_df.iloc[i]["Không Discount"]:.2f}', 
             ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

**Nhận định**:
- Từ biểu đồ grouped bar chart, ta thấy rằng trong hầu hết các danh mục sản phẩm, nhóm sản phẩm có giảm giá (discount_rate > 0) thường có *số lượng bán ra trung bình cao hơn nhiều* so với nhóm không giảm giá (discount_rate = 0). Điều này cho thấy rằng việc áp dụng giảm giá có thể thúc đẩy doanh số bán hàng trong nhiều danh mục sản phẩm.
- Một vài nhóm danh mục thậm chí luôn trong tình trạng giảm giá hoặc gần như vậy, điều này có thể phản ánh chiến lược giá của nhà bán lẻ trong các danh mục này.
- Quan sát được chỉ có danh mục **NGON** là số lượng bán ra trung bình bằng nhau giữa 2 nhóm có giảm giá và không giảm giá.

In [ ]:
# Kiểm định thống kê tương quan Spearman giữa price, discount_rate và quantity_sold cho từng danh mục
results = []
for category in categories:
    subset = df[df['category_root_name'] == category]
    
    if len(subset) < 30:
        continue  # Bỏ qua các danh mục có ít hơn 30 sản phẩm
    
    # Tương quan giữa price và quantity_sold
    corr_price_qty, p_value_price_qty = stats.spearmanr(subset['price'], subset['quantity_sold'])
    
    # Tương quan giữa discount_rate và quantity_sold
    corr_discount_qty, p_value_discount_qty = stats.spearmanr(subset['discount_rate'], subset['quantity_sold'])
    
    results.append({
        'Category': category,
        'Corr Price-Qty': corr_price_qty,
        'P-value Price-Qty': p_value_price_qty,
        'Corr Discount-Qty': corr_discount_qty,
        'P-value Discount-Qty': p_value_discount_qty
    })
results_df = pd.DataFrame(results)

# Tạo function để tô màu những p-value > 0.1
def highlight_non_significant(val):
    """
    Tô màu đỏ nhạt cho các p-value > 0.1 (không có ý nghĩa thống kê)
    """
    if isinstance(val, (int, float)):
        if val > 0.1:
            # Chữ đen nên xanh dương
            return 'background-color: #ff9999; color: #000000 ;font-weight: bold';  
        else:
            return ''
    return ''

print("\nKết quả kiểm định thống kê tương quan Spearman theo từng danh mục:")
print("(Các ô có màu đỏ nhạt là p-value > 0.1 - không có ý nghĩa thống kê)")

# Áp dụng styling cho DataFrame
styled_df = results_df.round(3).style.applymap(highlight_non_significant, 
                                               subset=['P-value Price-Qty', 'P-value Discount-Qty'])
display(styled_df)

**NHẬN ĐỊNH CHUNG và KIỂM ĐỊNH**:
- Qua các phân tích và biểu đồ trực quan hóa dữ liệu, chúng ta có thể kết luận rằng cả giá sản phẩm (price) và tỷ lệ giảm giá (discount_rate) đều ảnh hưởng đáng kể đến số lượng bán ra (quantity_sold) trong hầu hết các danh mục sản phẩm.
- Việc giảm giá sản phẩm thường dẫn đến tăng số lượng bán ra, trong khi giá sản phẩm cao hơn thường làm giảm số lượng bán ra. Tuy nhiên, mức độ ảnh hưởng của từng yếu tố có thể khác nhau tùy thuộc vào đặc thù của từng danh mục sản phẩm.
- Dựa vào các kết quả kiểm định thống kê, chúng ta có thể xác nhận rằng cả hai yếu tố price và discount_rate đều có ảnh hưởng đáng kể đến quantity_sold trong hầu hết các danh mục sản phẩm, với mức ý nghĩa thống kê phù hợp (p-value < 0.1).

***Một vài insight quan sát được***:
- Danh mục duy nhất có `quantity_sold` hoàn toàn không bị ảnh hưởng bởi `price` và `discount_rate` thường là **Điện thoại - Máy tính bảng**. Người dùng sẵn lòng chi trả cao cho các sản phẩm trong danh mục này bất kể giá cả hay giảm giá.
- Các danh mục như **Túi thời trang nữ, Nhà sách Tiki, Đồ chơi - Mẹ & Bé, Điện Gia Dụng** có `quantity_sold` không ảnh hưởng bởi `price` nhưng vẫn bị ảnh hưởng bởi `discount_rate`. Điều này cho thấy rằng người tiêu dùng trong các danh mục này có thể nhạy cảm hơn với các chương trình giảm giá so với giá cả sản phẩm. Cho thấy nếu cùng một mức giá giữa các shop thì shop nào có giảm giá sẽ bán được nhiều hơn.
- Cần chú ý các danh mục bị ảnh hưởng sâu sắc bởi `price` với hệ số tương quan âm cao như **Laptop – Máy Vi Tính, Điện Tử - Điện Lạnh và Ô Tô – Xe Máy** chứng tỏ người tiêu dùng trong các danh mục này rất nhạy cảm với giá cả, và việc tăng giá có thể dẫn đến giảm mạnh số lượng bán ra.
- Các danh mục còn lại đều tuân theo quy luật và cho thấy `quantity_sold` bị ảnh hưởng bởi cả `price` và `discount_rate`, với xu hướng chung là giá cao làm giảm số lượng bán ra, trong khi giảm giá làm tăng số lượng bán ra.

***Chiến lược đề xuất***:
- Đối với các dạnh mục **không nhạy cảm với cả giá cả và giảm giá**, nhà bán lẻ có thể duy trì chiến lược giá hiện tại mà không cần điều chỉnh nhiều về giá cả hay giảm giá, tập trung vào các yếu tố khác như chất lượng sản phẩm và dịch vụ khách hàng để tăng doanh số bán hàng.
- Đối với các danh mục **chỉ nhạy cảm với giá cả**, nhà bán lẻ nên cân nhắc kỹ lưỡng trước khi tăng giá sản phẩm để tránh giảm doanh số bán hàng. Nên tập trung vào việc duy trì mức giá cạnh tranh.
- Đối với các danh mục **chỉ nhạy cảm với giảm giá** , nhà bán lẻ nên tập trung vào việc triển khai các chương trình giảm giá hấp dẫn để thúc đẩy doanh số bán hàng. Không nên quá chú trọng vào việc điều chỉnh giá cả.
- Đối với các danh mục **nhạy cảm với cả giá cả và giảm giá**, nhà bán lẻ nên áp dụng chiến lược giá linh hoạt, kết hợp giữa việc duy trì mức giá hợp lý và triển khai các chương trình giảm giá định kỳ để tối ưu hóa doanh số bán hàng. 